
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# Troubleshooting DLT Python Syntax

Now that we've gone through the process of configuring and running a pipeline with 2 notebooks, we'll simulate developing and adding a 3rd notebook.

**DON'T PANIC!**

The code provided below contains some intentional, small syntax errors. By troubleshooting these errors, you'll learn how to iteratively develop DLT code and identify errors in your syntax.

This lesson is not meant to provide a robust solution for code development and testing; rather, it is intended to help users getting started with DLT and struggling with an unfamiliar syntax.

## Learning Objectives
By the end of this lesson, students should feel comfortable:
* Identifying and troubleshooting DLT syntax 
* Iteratively developing DLT pipelines with notebooks

## Add this Notebook to a DLT Pipeline

At this point in the course, you should have a DLT Pipeline configured with 2 notebook library.

You should have processed several batches of records through this pipeline, and should understand how to trigger a new run of the pipeline and add an additional library.

To begin this lesson, go through the process of adding this notebook to your pipeline using the DLT UI, and then trigger an update.

<img src="https://files.training.databricks.com/images/icon_hint_24.png"> The link to this notebook can be found back in [DE 4.1 - DLT UI Walkthrough]($../DE 4.1 - DLT UI Walkthrough)<br/>
in the printed instructions for **Task #3** under the section **Generate Pipline Configuration**

## Troubleshooting Errors

Each of the 3 functions below contains a syntax error, but each of these errors will be detected and reported slightly differently by DLT.

Some syntax errors will be detected during the **Initializing** stage, as DLT is not able to properly parse the commands.

Other syntax errors will be deteced during the **Setting up tables** stage.

Note that because of the way DLT resolves the order of tables in the pipeline at different steps, you may sometimes see errors thrown for later stages first.

An approach that can work well is to fix one table at a time, starting at your earliest dataset and working toward your final. Commented code will be ignored automatically, so you can safely remove code from a development run without removing it entirely.

Even if you can immediately spot the errors in the code below, try to use the error messages from the UI to guide your identification of these errors. Solution code follows in the cell below.

## Solutions

The correct syntax for each of our above functions is provided in a notebook by the same name in the Solutions folder.

To address these errors you have serveral options:
* Work through each issue, fixing the problems above yourself
* Copy and paste the solution in the **`# ANSWER`** cell from the Solutions notebook of the same name
* Update your pipline to directly use the Solutions notebook of the same name

**NOTE**: You won't be able to see any of the other errors until you add the **`import dlt`** statement to the cell above.

The issues in each query:
1. The **`@dlt.table`** decorator is missing before the function definition
1. The correct keyword argument to provide a custom table name is **`name`** not **`table_name`**
1. To perform a read on a table in the pipeline, use **`dlt.read`** not **`spark.read`**

In [0]:
import dlt
import pyspark.sql.functions as F

source = spark.conf.get("source")


@dlt.table
def status_bronze():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .load(f"{source}/status")
            .select(
                F.current_timestamp().alias("processing_time"), 
                F.input_file_name().alias("source_file"), 
                "*"
            )
    )

    
@dlt.table(
        name = "status_silver"
    )
@dlt.expect_or_drop("valid_timestamp", "status_timestamp > 1640995200")
def status_silver():
    return (
        dlt.read_stream("status_bronze")
            .drop("source_file", "_rescued_data")
    )

    
@dlt.table
def email_updates():
    return (
        dlt.read("status_silver").alias("a")
            .join(
                dlt.read("subscribed_order_emails_v").alias("b"), 
                on="order_id"
            ).select(
                "a.*", 
                "b.email"
            )
    )

## Summary

By reviewing this notebook, you should now feel comfortable:
* Identifying and troubleshooting DLT syntax 
* Iteratively developing DLT pipelines with notebooks


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>